In [1]:
pip install jax jaxlib torch torchvision matplotlib

# Conceptual Questions
1. Tracing based JIT compilation executes a function once with example inputs then records those operations in a graph and compiles the graph. Scripting based JIT compilation analyzes the source code and compiles it ahead of execution to give a full control flow graph.

In [2]:
# Example of python function that works with scripting but not tracing from lecture
import jax
import jax.numpy as jnp
# Fails with tracing because the shape
@jax.jit
def f (x , flag ) :
  if flag :
    return x * 2
  else :
    return x + 2

In [ ]:
@jax.jit
def process_batch ( data , batch_size ) :
  num_batches = data.shape[0] // batch_size
  results = []
  for i in range(num_batches ) :
    batch = data[ i * batch_size :( i + 1) * batch_size ]
    results.append (jnp.sum ( batch ** 2) )
  return jnp.array ( results )

*   The output shape is dynamic because num_batches depends on batch_size which may not be a static/semi-static value
* Python lists inside JIT-compiled functions requires JAX to trace and unroll the loop to create large computational graphs
* Python loops execute during tracing and get unrolled, wheare jax control flow primitives keep loops inside the graph.






In [6]:
import jax
import jax.numpy as jnp

def process_batch(data, batch_size):
    num_batches = data.shape[0] // batch_size
    batches = data[:num_batches * batch_size].reshape(num_batches, batch_size)
    return jax.vmap(lambda b: jnp.sum(b ** 2))(batches)

process_batch = jax.jit(process_batch, static_argnums=(1,))

# Problem 1.3


1.   In eager mode each operation launches its own kernel . There are 4 total operations and 5 global reads and 4 writes for a total of 9 operations
2.  In fused mode all operations execute on 1 kernel and stores the intermediates in registers meaning there are 2 memory operations. One to read input and one to write output to
3. Memory bandwith reduction factor = eager memory ops / fused memory ops so The memory bandwith factor is 4.5


# Problem 1.4

Dynamic control flow poses challenges for JIT compilation because the compiler only has access to symbolic tensors giving it access to the shape and data types, but not the actual values. For example control flow statement like if x > 2 depends on runtime values the compiler cannot determine which branch to include in the static graph which can cause an error. JAX solves this problem by requiring explicit control flow operators such as lax.cond which will compile both branches and select one at run time. Pytorch instead addresses the problem of control flow in multiple ways such as graph breaks, recompilation, and using control flow operators like torch.cond(similair to JAX) to compile both branches.


1.   Graph breaks: by default returns to eager execution at dynamic branching points at the cost of syncing the GPU and CPU(can decrease performance)
2. Recompilation: attatches guards to compiled graphs to mark the branch taken and are cached.(can decrease performance if the condition changes every call it will recompile frequently)
3. Control flow operators: compile multiple branches in a static graph and selecting branch at runtime


# Problem 1.5

When torch.compile is run on a function for the first time it needs to build an optimized version of the code, which has added overhead. First TorchDynamo intercepts Python execution and traces it into FX graph using fake tensors. Then the graph is lowered through decomposition and AOTAutograd. Then TorchInductor compiles the graph into Triton/CUDA kernels which are then cached, making subsequent calls faster.